# SVD and PCA

| Difficulty | ████░░░░░░ 4/10 |
| :--- | :--- |
| Prerequisites | 01.06 |
| Time | ~75 min |
| Colab GPU | Not needed |

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NiravRVaghasiya/maths-for-ai/blob/main/01_Linear_Algebra/07_SVD_and_PCA.ipynb)

---

## 🎯 Learning Objective

Derive the Singular Value Decomposition as the generalization of eigendecomposition to any matrix, and use it to perform Principal Component Analysis for dimensionality reduction.

---

## 📐 Theory

Eigendecomposition requires a square matrix. The **Singular Value Decomposition (SVD)**
generalizes the idea to *any* matrix $A$ (shape $m\times n$):

$$A = U\Sigma V^\top$$

- $U$ ($m\times m$) and $V$ ($n\times n$) are orthogonal (their columns are orthonormal).
- $\Sigma$ ($m\times n$) is diagonal with non-negative entries $\sigma_1 \ge \sigma_2 \ge
  \cdots \ge 0$, the **singular values**.

Geometrically: *any* linear map is "rotate ($V^\top$), scale along axes ($\Sigma$), rotate
again ($U$)." This decomposes even non-square, rank-deficient, or asymmetric matrices — the
generalization eigendecomposition couldn't offer.

### Connection to eigendecomposition

The singular values of $A$ are the square roots of the eigenvalues of $A^\top A$ (which is
always symmetric positive-semi-definite, so this is always well-defined even when $A$ itself
isn't square or symmetric).

### Truncated SVD and the Eckart-Young theorem

Keeping only the top $k$ singular values/vectors gives $A_k = U_k \Sigma_k V_k^\top$, the
*best possible* rank-$k$ approximation of $A$ in a precise sense (proved by the Eckart-Young
theorem, revisited in [Notebook 08.05](../08_Advanced_Linear_Algebra/05_low_rank_approximations.ipynb)).
The fraction of "energy" (squared Frobenius norm) captured by the top $k$ components is
$\sum_{i\le k}\sigma_i^2 \big/ \sum_i \sigma_i^2$ — this is exactly the "variance explained" number
PCA reports.

### PCA via SVD

Running SVD directly on the *centered data matrix* $X$ (rather than eigendecomposing the
covariance matrix, as in [Notebook 01.06](06_eigenvalues_eigenvectors.ipynb)) is the
numerically preferred way to compute PCA in practice, since it avoids ever forming
$X^\top X$ explicitly (which can lose precision).

---

## 👁️ Visual Intuition

Reconstructing an image from a truncated SVD, rank by rank, makes "low-rank approximation"
visceral: a handful of singular values already captures most of the recognizable structure.

In [ ]:
# Truncated SVD image compression: how few singular values are needed to recognize the image?
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline
plt.style.use('seaborn-v0_8-whitegrid')

# Build a simple synthetic "image": concentric rings (has real low-rank structure to exploit)
size = 80
yy, xx = np.mgrid[0:size, 0:size]
center = size / 2
radius = np.sqrt((xx - center) ** 2 + (yy - center) ** 2)
image = np.sin(radius / 3.0) * np.exp(-radius / 60.0)

U, S, Vt = np.linalg.svd(image, full_matrices=False)

ranks = [1, 5, 20, size]
fig, axes = plt.subplots(1, len(ranks), figsize=(14, 4))
for ax, k in zip(axes, ranks):
    approx = U[:, :k] @ np.diag(S[:k]) @ Vt[:k, :]
    energy_captured = (S[:k] ** 2).sum() / (S ** 2).sum()
    ax.imshow(approx, cmap="viridis")
    ax.set_title(f"rank {k}\n{energy_captured*100:.1f}% energy")
    ax.axis("off")
plt.tight_layout()
plt.show()

## 🐍 Implementation from Scratch

We compute the SVD via eigendecomposition of $A^\top A$ (the textbook definition), confirm it
matches `np.linalg.svd`, then run PCA via SVD directly on centered data.

In [ ]:
# SVD from the eigendecomposition of A^T A, then PCA via SVD on centered data
import numpy as np

def svd_from_scratch(A):
    # Singular values/right-singular-vectors come from eigendecomposing the symmetric A^T A
    AtA = A.T @ A
    eigvals, V = np.linalg.eigh(AtA)
    order = np.argsort(eigvals)[::-1]
    eigvals, V = eigvals[order], V[:, order]
    eigvals = np.clip(eigvals, 0, None)         # guard against tiny negative numerical noise
    singular_values = np.sqrt(eigvals)

    # Left-singular vectors: U_i = A v_i / sigma_i  (for nonzero sigma_i)
    U = np.zeros((A.shape[0], len(singular_values)))
    for i, sv in enumerate(singular_values):
        if sv > 1e-10:
            U[:, i] = (A @ V[:, i]) / sv
    return U, singular_values, V.T

A = np.array([[3.0, 1.0], [1.0, 3.0], [1.0, 1.0]])
U_scratch, S_scratch, Vt_scratch = svd_from_scratch(A)
U_np, S_np, Vt_np = np.linalg.svd(A, full_matrices=False)

print("Singular values (scratch):", np.round(S_scratch, 4))
print("Singular values (numpy)  :", np.round(S_np, 4))
reconstructed = U_scratch @ np.diag(S_scratch) @ Vt_scratch
print("Reconstruction A ~ U S V^T matches original A:", np.allclose(reconstructed, A))

print("\n--- PCA via SVD directly on centered data ---")
rng = np.random.default_rng(3)
X = rng.multivariate_normal([0, 0], [[3.0, 1.5], [1.5, 1.0]], size=500)
X_centered = X - X.mean(axis=0)

U, S, Vt = np.linalg.svd(X_centered, full_matrices=False)
explained_variance = (S ** 2) / (X.shape[0] - 1)
print("Explained variance via SVD:", np.round(explained_variance, 3))
print("(matches the eigenvalues of the covariance matrix computed in Notebook 01.06)")

## 🤖 Why This Matters for AI

**LoRA (Low-Rank Adaptation)** fine-tunes large language models by freezing the pretrained
weight matrix $W_0$ and learning only a low-rank *update*:

$$W = W_0 + BA, \qquad B \in \mathbb{R}^{d\times r},\ A \in \mathbb{R}^{r\times d},\ r \ll d$$

This is the SVD's low-rank approximation idea turned into a training strategy: instead of
updating all $d^2$ parameters of $W$, you only learn $2dr$ parameters — often a >100x
reduction. We simulate this below: approximate a "full fine-tuning" weight update with its
best rank-$r$ approximation and measure how much of the update's effect survives.

In [ ]:
# Simulating LoRA: approximating a full weight update with a low-rank factorization
import numpy as np

rng = np.random.default_rng(0)
d = 64  # a stand-in for a hidden dimension (real LLMs: thousands)

W0 = rng.standard_normal((d, d)) * 0.02              # frozen pretrained weights
delta_W_full = rng.standard_normal((d, d)) * 0.01     # what "full fine-tuning" would have learned

U, S, Vt = np.linalg.svd(delta_W_full)

for r in [1, 4, 16, 64]:
    B = U[:, :r] * S[:r]           # (d, r)
    A_lora = Vt[:r, :]              # (r, d)
    delta_W_lora = B @ A_lora        # rank-r approximation of the full update

    full_params = d * d
    lora_params = d * r + r * d
    energy_captured = (S[:r] ** 2).sum() / (S ** 2).sum()
    approx_error = np.linalg.norm(delta_W_full - delta_W_lora) / np.linalg.norm(delta_W_full)

    print(f"rank r={r:3d}: {lora_params:6d}/{full_params} params "
          f"({100*lora_params/full_params:5.1f}%), "
          f"energy captured={energy_captured*100:5.1f}%, "
          f"relative approx error={approx_error:.3f}")

## 🏋️ Exercises

Six levels, in order: mechanical computation → conceptual understanding → derivation →
implementation → application → open-ended challenge. Solutions/outlines are in the collapsed
`Solution` blocks below each exercise — try before revealing.

### 🔢 COMPUTE
1. Compute the SVD of $\begin{pmatrix}3&0\\0&-2\end{pmatrix}$ by hand (note: singular values are
   always non-negative, even though this matrix's eigenvalues are $3$ and $-2$). Verify with
   `np.linalg.svd`, and confirm both $U$ and $V$ come out orthogonal ($U^\top U=I$,
   $VV^\top=I$) — a structural guarantee of the SVD for *any* matrix, not something specific to
   this diagonal example.

<details>
<summary>💡 Solution</summary>

Since $A$ is already diagonal with one negative entry, the SVD just needs to absorb that sign:
$\sigma_1=3,\sigma_2=2$ (sorted, non-negative), $U=I$, and $V=\begin{pmatrix}1&0\\0&-1\end{pmatrix}$
(the sign flip moves into $V$, so that $U\Sigma V^\top$ still reproduces the $-2$: $1\cdot2\cdot(-1)=-2$).
This matches `np.linalg.svd(A)` exactly. Both $U^\top U$ and $VV^\top$ equal the identity to
floating-point precision — guaranteed for *any* $A$'s SVD, since $U$ and $V$ are defined to be
orthogonal matrices, unlike eigendecomposition's $V$ in $A=V\Lambda V^{-1}$, which has no such
guarantee for a general (non-symmetric) matrix.

</details>

### 💭 UNDERSTAND
2. For a *symmetric* matrix, singular values equal the absolute values of the eigenvalues —
   true, and easy to check on the notebook's own $\begin{pmatrix}3&0\\0&-2\end{pmatrix}$ example
   above ($|3|=3,|-2|=2$, exactly its singular values). Does this same relationship
   ($\sigma_i=|\lambda_i|$, matched up correctly) hold for a general, non-symmetric square
   matrix? Test it on the nilpotent matrix $N=\begin{pmatrix}0&2\\0&0\end{pmatrix}$ (whose only
   eigenvalue, with multiplicity 2, is $0$) before deciding.

<details>
<summary>💡 Solution</summary>

No — it's specific to symmetric matrices, and $N$ is a clean counterexample: $N$'s eigenvalues
are both exactly $0$ (its characteristic polynomial is $\lambda^2=0$), but its singular values
are $2$ and $0$ — $N$ visibly does *not* collapse every vector to zero ($N(1,0)=(0,0)$ but
$N(0,1)=(2,0)$), so it can't have all-zero singular values, even though both its eigenvalues
are zero. More generally, singular values come from $\sqrt{\text{eigenvalues of }A^\top A}$,
while eigenvalues come from $A$ directly — for a non-symmetric $A$, $A^\top A$ mixes $A$ with
its own transpose in a way that has no simple relationship to $A$'s own eigenvalues (checked
also on `01.06`'s own $\begin{pmatrix}4&1\\2&3\end{pmatrix}$, eigenvalues exactly $5,2$ but
singular values $\approx5.117,1.954$ — close but not equal, confirming the relationship isn't
even "true up to some simple correction" in general). Symmetry is exactly what collapses this
distinction: for symmetric $A$, $A^\top A=A^2$, whose eigenvalues are exactly the squares of
$A$'s own eigenvalues, which is the entire reason $\sigma_i=|\lambda_i|$ works out there and
nowhere else.

</details>

### ✏️ DERIVE
3. Derive that the sum of squared singular values equals the squared Frobenius norm,
   $\|A\|_F^2=\sum_i\sigma_i^2$, starting from $\|A\|_F^2=\text{tr}(A^\top A)$ and the SVD
   $A=U\Sigma V^\top$.

<details>
<summary>💡 Solution outline</summary>

Start from the trace identity and substitute the SVD:
$$\|A\|_F^2 = \text{tr}(A^\top A) = \text{tr}\big((U\Sigma V^\top)^\top(U\Sigma V^\top)\big) =
\text{tr}(V\Sigma^\top U^\top U\Sigma V^\top).$$
Since $U$ is orthogonal, $U^\top U=I$, collapsing this to $\text{tr}(V\Sigma^\top\Sigma V^\top)$.
Using the cyclic property of trace, $\text{tr}(V\Sigma^\top\Sigma V^\top) =
\text{tr}(\Sigma^\top\Sigma V^\top V) = \text{tr}(\Sigma^\top\Sigma)$ (since $V^\top V=I$ too).
Finally, $\Sigma^\top\Sigma$ is diagonal with entries $\sigma_i^2$, so its trace is exactly
$\sum_i\sigma_i^2$. Numeric check on a random $4\times3$ matrix: $\|A\|_F^2\approx9.688$ and
$\sum_i\sigma_i^2\approx9.688$ match to floating-point precision, and both equal
$\text{tr}(A^\top A)$ computed directly, three routes to the same number. This identity is
exactly what makes "energy captured" ($\sum_{i\le k}\sigma_i^2/\sum_i\sigma_i^2$) a meaningful
fraction: the denominator is literally $\|A\|_F^2$, the total squared magnitude of $A$.

</details>

### 🐍 IMPLEMENT
4. Write `pca_reconstruct(X, k)` that centers `X`, computes its SVD, keeps only the top $k$
   components, and returns the reconstructed (un-centered) data. Test it on the notebook's own
   correlated 2D `X`, checking reconstruction error at $k=0$ (reconstruct with only the mean),
   $k=1$, and $k=2$ (full rank).

<details>
<summary>✅ How to know you're right</summary>

At $k=2$ (the data's full rank, 2 dimensions), your reconstruction must match the original `X`
exactly (up to floating-point rounding) — keeping *every* singular value/vector discards
nothing, so any nonzero error at full rank means a bug. The Frobenius-norm reconstruction error
should decrease monotonically as $k$ increases from 0 to 2 (e.g. roughly $44.8\to10.1\to0.0$ for
one sample draw of the notebook's `X`) — never increasing, since each additional component
provides more information, and the SVD orders components by exactly how much reconstruction
error they resolve, largest first.

</details>

### 🤖 APPLY
5. A noisy $40\times30$ data matrix is secretly $A_{\text{true}}+\text{noise}$, where
   $A_{\text{true}}$ has true rank 3 (built as a product of two rank-3 factors) and the noise is
   i.i.d. Gaussian. Denoise it by keeping only the top-3 truncated SVD of the *noisy* matrix, and
   compare $\|A_{\text{true}} - A_{\text{noisy}}\|_F$ against $\|A_{\text{true}} -
   A_{\text{denoised}}\|_F$ (both relative to $\|A_{\text{true}}\|_F$) across several noise
   levels.

<details>
<summary>✅ What you should observe</summary>

Truncated-SVD denoising should reduce the relative error substantially and consistently across
every noise level tested — for example, at noise scale $1.0$ (comparable in magnitude to the
signal itself), the raw noisy matrix has relative error $\approx3.42$, while the rank-3
denoised version cuts that to $\approx1.83$ (roughly half), and this pattern holds at much
smaller noise (relative error $\approx0.035\to0.014$) and much larger noise (relative error
$\approx34.3\to18.0$) alike, and across different random noise draws. This works because random
noise spreads its energy across *all* singular directions roughly evenly, while
$A_{\text{true}}$'s energy concentrates in just 3 — keeping only the top 3 singular
components keeps almost all of the signal while discarding most of the noise, along the same
logic as the image-compression demo in this notebook's own Visual Intuition cell, just framed as
denoising a data matrix rather than compressing an image.

</details>

### 🚀 CHALLENGE
6. In the denoising setup above, using the *correct* rank ($k=3$, matching $A_{\text{true}}$'s
   actual rank) isn't automatic — you have to choose $k$. Investigate what happens to
   reconstruction error as you vary $k$ away from 3 (try $k=1,2,3,4,5,8,15,30$) at a fixed,
   moderate noise level. Is $k=3$ actually the best choice, and if so, what causes the error to
   go back *up* for $k>3$ — shouldn't keeping more components always capture at least as much of
   $A_{\text{true}}$?

<details>
<summary>🔍 What a strong answer covers</summary>

A strong answer finds a genuine U-shaped (or J-shaped) error curve at moderate noise (e.g. noise
scale $0.1$–$0.2$, where signal and noise are comparable but not identical in magnitude): error
decreases sharply from $k=1$ to $k=3$ (e.g. $\approx0.68\to0.43\to0.14$ at noise scale $0.1$),
reaches its minimum almost exactly at the true rank $k=3$, and then *increases* again for
$k=4,5,8,\ldots$ (rising to $\approx0.35$ by $k=30$, keeping *every* component) — confirmed
across several noise levels, with $k=3$ consistently the best or near-best choice. The
resolution to the apparent paradox: components beyond the true rank aren't "extra signal you're
missing out on" — $A_{\text{true}}$ genuinely has no content there, so singular vectors past
index 3 are capturing *pure noise*, and including them in the reconstruction adds that noise
back in rather than removing it. A strong answer connects this to the real-world version of the
same problem: choosing $k$ (or a LoRA rank $r$, or a PCA component count) always trades
underfitting (too few components, missing real signal) against overfitting to noise (too many
components, reconstructing noise as if it were signal) — the "elbow" in an energy-captured or
reconstruction-error curve is a heuristic for finding that sweet spot, not a guarantee, since in
a real dataset (unlike this synthetic one) the true underlying rank is never known exactly.

</details>

---

## 📚 Further Reading
- Hu et al., ["LoRA: Low-Rank Adaptation of Large Language Models"](https://arxiv.org/abs/2106.09685)
- Strang, *Introduction to Linear Algebra*, Ch. 7 (Singular Value Decomposition)

---

*Next notebook: [Matrix Decompositions](08_matrix_decompositions.ipynb)*